# Image Colourisation: GAN Comparison
Implementation of a **Conditional GAN** (U-Net + PatchGAN) for 256x256 images using the COCO dataset.

This notebook compares two output strategies:
1. **Regression**: Predicting continuous AB values (L1 Loss).
2. **Classification**: Predicting discrete colour bins (Cross-Entropy with Rebalancing).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import torch
import torchvision.transforms as T
from train import train_gan_loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# --- Configuration ---
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 10
MAX_SAMPLES = 1000 # Number of images to use from the dataset

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor()
])

# Dataset Path Detection
base_path = "data/coco" if os.path.exists("data/coco/train") else "../data/coco"
if not os.path.exists(os.path.join(base_path, "train")):
    print("ERROR: Dataset not found. Please run 'python scripts/setup_coco.py' first.")
else:
    print(f"Dataset found at: {base_path}")

## 1. GAN + Regression (L1 Loss)
Directly predicts the `a` and `b` channels as continuous values in range `[-1, 1]`.

In [ ]:
netG_reg, netD_reg = train_gan_loop(
    base_path=base_path,
    transform=transform,
    device=device,
    mode='regression',
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS
)

## 2. GAN + Classification (Cross-Entropy)
Predicts one of 100 colour bins for each pixel. Includes class rebalancing to favour vibrant colours.

In [ ]:
netG_cls, netD_cls = train_gan_loop(
    base_path=base_path,
    transform=transform,
    device=device,
    mode='classification',
    use_rebalancing=True,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS
)